In [8]:
import os
import glob
import numpy as np
import pandas as pd

def run_prospective_volatility_test(csv_path, psi_threshold=0.35, window=3):
    """
    Simulates a prospective early stopping trigger based on the rolling standard deviation of Psi.
    """
    if not os.path.exists(csv_path):
        return None
        
    df = pd.read_csv(csv_path)
    filename = os.path.basename(csv_path)
    
    # Compute rolling standard deviation (Ψ-volatility)
    df['Psi_volatility'] = df['Psi'].rolling(window=window).std()
    
    # Simulate prospective early stopping trigger
    psi_stop_epoch = None
    consecutive_count = 0
    
    for idx, row in df.iterrows():
        vol = row['Psi_volatility']
        if not np.isnan(vol) and vol < psi_threshold:
            consecutive_count += 1
        else:
            consecutive_count = 0
            
        if consecutive_count >= window:
            psi_stop_epoch = int(row['epoch'])
            break
            
    actual_stop_epoch = int(df['epoch'].max())
    max_val_acc = df['val_acc'].max()
    
    if psi_stop_epoch is not None:
        acc_at_psi_stop = df[df['epoch'] == psi_stop_epoch]['val_acc'].values[0]
        epochs_saved = actual_stop_epoch - psi_stop_epoch
        acc_delta = max_val_acc - acc_at_psi_stop
        status = "TRIGGERED"
    else:
        psi_stop_epoch = actual_stop_epoch
        acc_at_psi_stop = df['val_acc'].iloc[-1]
        epochs_saved = 0
        acc_delta = max_val_acc - acc_at_psi_stop
        status = "FAILED"
        
    return {
        "filename": filename,
        "status": status,
        "actual_stop_epoch": actual_stop_epoch,
        "psi_stop_epoch": psi_stop_epoch,
        "epochs_saved": epochs_saved,
        "acc_at_psi_stop": acc_at_psi_stop,
        "acc_delta_vs_max": acc_delta
    }

def process_results_directory(directory_path, psi_threshold=0.35, window_size=3):
    """
    Scans a targeted directory for results_*.csv files and executes the validation protocol.
    """
    if not os.path.exists(directory_path):
        print(f"[-] Directory not found: {directory_path}")
        return
        
    csv_files = glob.glob(os.path.join(directory_path, "results_*.csv"))
    if not csv_files:
        print(f"[-] No 'results_*.csv' files discovered in: {directory_path}")
        return
        
    results_list = []
    for file_path in csv_files:
        res = run_prospective_volatility_test(file_path, psi_threshold, window_size)
        if res:
            results_list.append(res)
            
    # Print formatted clean summary table
    print(f"{'Log File Name':<32} | {'Status':<11} | {'Stop Epoch (Loss)':<18} | {'Stop Epoch (Ψ)':<14} | {'Saved Ep.':<10} | {'Acc at Ψ (%)':<14} | {'Δ Acc vs Max':<12}")
    print("-" * 115)
    
    for r in results_list:
        print(f"{r['filename']:<32} | {r['status']:<11} | {r['actual_stop_epoch']:<18d} | {r['psi_stop_epoch']:<14d} | {r['epochs_saved']:<10d} | {r['acc_at_psi_stop']:<14.2f} | {r['acc_delta_vs_max']:<12.2f}")

In [11]:
# ===========================================================================
# EXPERIMENT 1: CIFAR-10 TRAJECTORIES (OPTIMIZED ADAPTIVE PARAMETERS)
# ===========================================================================
cifar10_dir = "results/cifar10"

print("=" * 115)
print(f"🔬 ADAPTIVE PROSPECTIVE THRESHOLD EXPERIMENT: CIFAR-10 DATASET")
print("=" * 115)

process_results_directory(cifar10_dir, psi_threshold=0.40, window_size=3)
print("=" * 115)

🔬 ADAPTIVE PROSPECTIVE THRESHOLD EXPERIMENT: CIFAR-10 DATASET
Log File Name                    | Status      | Stop Epoch (Loss)  | Stop Epoch (Ψ) | Saved Ep.  | Acc at Ψ (%)   | Δ Acc vs Max
-------------------------------------------------------------------------------------------------------------------
results_vgg16_cifar10.csv        | TRIGGERED   | 26                 | 16             | 10         | 79.65          | 3.53        
results_resnet34_cifar10.csv     | TRIGGERED   | 27                 | 23             | 4          | 72.77          | 4.03        
results_resnet50_cifar10.csv     | TRIGGERED   | 40                 | 26             | 14         | 68.60          | 5.16        
results_resnet18_cifar10.csv     | TRIGGERED   | 25                 | 25             | 0          | 77.66          | 0.95        
results_resnet152_cifar10.csv    | FAILED      | 38                 | 38             | 0          | 68.64          | 0.00        
results_mobilenet_cifar10.csv    | TRIGGER

In [15]:
# ===========================================================================
# EXPERIMENT 2: CIFAR-100 TRAJECTORIES
# ===========================================================================
cifar100_dir = "results/cifar100"

print("\n" + "=" * 115)
print(f"🔬 PROSPECTIVE THRESHOLD EXPERIMENT: CIFAR-100 DATASET")
print("=" * 115)

# Khởi chạy phân tích thư mục CIFAR-100 với cùng cấu hình quy tắc cố định
process_results_directory(cifar100_dir, psi_threshold=0.20, window_size=3)
print("=" * 115)


🔬 PROSPECTIVE THRESHOLD EXPERIMENT: CIFAR-100 DATASET
Log File Name                    | Status      | Stop Epoch (Loss)  | Stop Epoch (Ψ) | Saved Ep.  | Acc at Ψ (%)   | Δ Acc vs Max
-------------------------------------------------------------------------------------------------------------------
results_resnet101_cifar100.csv   | TRIGGERED   | 58                 | 50             | 8          | 46.97          | 2.16        
results_resnet34_cifar100.csv    | TRIGGERED   | 48                 | 27             | 21         | 48.70          | 8.32        
results_resnet152_cifar100.csv   | TRIGGERED   | 68                 | 49             | 19         | 36.06          | 17.33       
results_mobilenet_cifar100.csv   | FAILED      | 50                 | 50             | 0          | 45.44          | 0.91        
results_vgg16_cifar100.csv       | TRIGGERED   | 55                 | 27             | 28         | 56.05          | 7.77        
results_resnet18_cifar100.csv    | TRIGGERED   | 